### 끝말잇기 게임

**멀티턴** 대화 관리 기술을 실제 게임에 적용한 예시

#### 게임 규칙
1. 이미 나온 단어를 다시 말하면 패배
2. 두음법칙 허용 (ex. 리→이, 력→역, 락→낙)
3. 국어사전에 존재하는 명사만 허용
4. **아무런 설명 없이, 끝말잇기 단어만 한글로 한 단어만 출력** (프롬프트에 명확히 지시)

#### 멀티턴 대화와의 연관성
- AI가 **이전 턴에 나온 단어들을 기억**해야 하므로, 멀티턴 대화 관리 기술이 필수적입니다.
- 대화 기록을 바탕으로 **중복 단어 방지, 규칙 위반 감지**가 가능합니다.
- 실제 서비스에서는 세션별로 기록을 분리해 여러 사용자가 동시에 게임을 즐길 수 있습니다.

#### 실습 포인트
- 프롬프트 설계가 중요: "설명 없이 단어만 출력"을 명확히 지시해야 LLM이 불필요한 설명을 하지 않음

In [5]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [ ]:
system_prompt="""당신은 끝말잇기 게임을 진행하는 AI 챗봇입니다.
아래 순서가 있는 목록은 게임 규칙입니다.
당신과 user 의 입력에서 아래 규칙을 반드시 지켜야 합니다.
규칙을 지키지 않은 사람에게 패배를 알린 뒤, 끝말잇기 게임을 종료합니다.
user가 말한 단어의 끝말로 시작하지 못하면 당신의 패배입니다.
user가 승리한 경우에는 "YOU WIN!", user가 패배한 경우에는 "YOU LOSE!"를 출력합니다.

1. 주어진 대화 기록에서 이미 나왔던 단어를 다시 말했을 경우 패배합니다.
2. 두음법칙을 허용합니다. (ex. 리 -> 이, 력 -> 역, 락 -> 낙)
3. 국어사전에 존재하는 단어이자, 명사여야 합니다.
4. 한 글자 단어는 사용하지 않습니다.
5. 아무런 설명 없이, 끝말잇기 단어만 한글로 한 단어만 출력하세요."""

In [32]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver  

agent = create_agent(
    model="google_genai:gemini-2.5-flash", 
    tools=[],
    checkpointer=InMemorySaver(),
    system_prompt=system_prompt,
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [33]:
user_input = input("🧑 YOUR TURN : ")

response = agent.invoke(
                {"messages": [{"role": "user", "content": user_input}]},
                {"configurable": {"thread_id": "1"}},  
            )

In [34]:
response["messages"][-1].content

'YOU LOSE!'

In [15]:
while True:
    user_input = input("🧑 YOUR TURN : ")
    if user_input == "종료": break
    response = agent.invoke(
                {"messages": [{"role": "user", "content": user_input}]},
                {"configurable": {"thread_id": "1"}},  
            )
    print("🤖 AI TURN : ", response["messages"][-1].content)

🤖 AI TURN :  도토리
🤖 AI TURN :  빨대
🤖 AI TURN :  포도
🤖 AI TURN :  포수
🤖 AI TURN :  도장
🤖 AI TURN :  독수리
🤖 AI TURN :  불꽃


In [5]:
from langchain_core.runnables import RunnablePassthrough

def summarize_messages(chain_input):
    stored_messages = chat_history.messages
    if len(stored_messages) == 0:
        return False
    summarization_prompt = ChatPromptTemplate.from_messages(
        [
            ("placeholder", "{chat_history}"),
            (
                "user",
                "위 채팅 메시지는 끝말잇기 게임을 진행한 대화내용입니다. 언급한 단어들만 나열하여 저장해주세요.",
            ),
        ]
    )
    summarization_chain = summarization_prompt | llm

    # chat_history 에 저장된 대화 기록을 요약프롬프트에 입력 & 결과 저장
    summary_message = summarization_chain.invoke({"chat_history": stored_messages})

    # chat_history 에 저장되어있던 기록 지우기
    chat_history.clear()

    # 생성된 새로운 요약내용으로 기록 채우기
    chat_history.add_message(summary_message)

    return True

chain_with_summarization = (
    # RunnablePassthrough는 LCEL에서 사용, 입력값을 다음 단계로 그대로 통과시키는 역할
    # assign() 메서드는 체인에 들어오는 딕셔너리에 새로운 키-값 추가
    RunnablePassthrough.assign(messages_summarized=summarize_messages) # 새로운 키 messages_summarized에 값(True 또는 False) 할당, 이 값은 조건부 요약에 활용 가능
    | chain_with_message_history
)

In [6]:
while True:
    user_input = input("🧑 YOUR TURN : ")
    if user_input == "종료": break
    response = chain_with_summarization.invoke(
                {"input": user_input},
                {"configurable": {"session_id": "unused"}},
            )
    print("🤖 AI TURN : ", response.content) # AIMessage 객체에서 .content 추출

🤖 AI TURN :  밥상
🤖 AI TURN :  장미


#### Gradio 챗봇

In [12]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv
load_dotenv(override=True)

gemini_api_key = os.getenv("GEMINI_API_KEY")

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=gemini_api_key)


# 대화 기록을 저장할 히스토리 클래스 불러오기
chat_history = ChatMessageHistory()

chat_history.messages

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """당신은 끝말잇기 게임을 진행하는 AI 챗봇입니다. 아래는 게임 규칙입니다. 당신과 user 의 입력에서 아래 규칙이 꼭 지켜져야 하며, 지키지 않은 사람에게 패배를 알린 뒤, 끝말잇기 게임을 종료합니다.
                1. 주어진 대화 기록에서 이미 나왔던 단어를 다시 말했을 경우 패배합니다.
                2. 두음법칙을 허용합니다. (ex. 리 -> 이, 력 -> 역, 락 -> 낙)
                3. 국어사전에 존재하는 단어이자, 명사여야 합니다.
                4. 아무런 설명 없이, 끝말잇기 단어만 한글로 한 단어만 출력하세요.
            """,
        ),
        ("placeholder", "{chat_history}"),
        ("user", "{input}"),
    ]
)

chain = prompt | llm

chain_with_message_history = RunnableWithMessageHistory(
    chain,
    lambda session_id: chat_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

def summarize_messages(chain_input):
    stored_messages = chat_history.messages
    if len(stored_messages) == 0:
        return False
    summarization_prompt = ChatPromptTemplate.from_messages(
        [
            ("placeholder", "{chat_history}"),
            (
                "user",
                "위 채팅 메시지는 끝말잇기 게임을 진행한 대화내용입니다. 언급한 단어들만 나열하여 저장해주세요.",
            ),
        ]
    )
    summarization_chain = summarization_prompt | llm

    # chat_history 에 저장된 대화 기록을 요약프롬프트에 입력 & 결과 저장
    summary_message = summarization_chain.invoke({"chat_history": stored_messages})

    # chat_history 에 저장되어있던 기록 지우기
    chat_history.clear()

    # 생성된 새로운 요약내용으로 기록 채우기
    chat_history.add_message(summary_message)

    return True

chain_with_summarization = (
    # RunnablePassthrough는 LCEL에서 사용, 입력값을 다음 단계로 그대로 통과시키는 역할
    # assign() 메서드는 체인에 들어오는 딕셔너리에 새로운 키-값 추가
    RunnablePassthrough.assign(messages_summarized=summarize_messages) # 새로운 키 messages_summarized에 값(True 또는 False) 할당, 이 값은 조건부 요약에 활용 가능
    | chain_with_message_history
)

In [13]:
import gradio as gr

def word_chain_response(message, history):
    """끝말잇기 게임 응답 함수 - 수정된 버전"""
    response = chain_with_summarization.invoke(
                {"input": message},  # user_input -> message로 수정
                {"configurable": {"session_id": "unused"}},
            )
    return response.content  # AIMessage 객체에서 .content 추출

demo = gr.ChatInterface(
    word_chain_response, 
    type="messages", 
    autofocus=False,
    title="🗣️ 끝말잇기 게임",
    description="AI와 함께 끝말잇기 게임을 해보세요! 단어만 입력하면 됩니다."
)

if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
